In [5]:
%pip install pandas
%pip install alpaca-py
%pip install pytz


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

In [1]:
'''
VERSION 1
import pandas as pd

# Load Excel
df = pd.read_excel("test4stocks.xlsx") #can be changed to whatever type we want 
df = pd.read_parquet("1000Stocks.parquet")
df = df.dropna() # clean empty lines
df["date"] = pd.to_datetime(df["date"]) # converts data from the fprm of YYYY MM DD Time type to pandas  

final_set = [] # empty list for dollar bars


for ticker, ticker_df in df.groupby("ticker"):  #split per tickers --- apple, nvda etc
    for date, day_df in ticker_df.groupby("date"):  #similarly split by day
        day_df = day_df.sort_values("date").reset_index(drop=True) # making sure the chronological order is fine and every piece starts as 0 index

    #Dollar Volume
        day_df["dollarVolume"] = day_df["close"] * day_df["volume"]

        # 50 bars per day target
        Strategist_Threshold = day_df["dollarVolume"].sum() / 50    #calculate the total dv per day and /50

        current_dollar_value = 0
        current_set = []

        #dollar bar
        for index, row in day_df.iterrows():

            dollar_tot = row["dollarVolume"]
            current_dollar_value += dollar_tot
            current_set.append(row)

            if current_dollar_value >= Strategist_Threshold:

                new_df_dollar_bar = pd.DataFrame(current_set) # new dataframe

                ticker_name = new_df_dollar_bar.iloc[0]["ticker"]
                open_price = new_df_dollar_bar.iloc[0]["close"]
                close_price = new_df_dollar_bar.iloc[-1]["close"]
                high_price = new_df_dollar_bar["close"].max()
                low_price = new_df_dollar_bar["close"].min()

                final_set.append({
                    "ticker": ticker_name,
                    "date": date,
                    "open": open_price,
                    "high": high_price,
                    "low": low_price,
                    "close": close_price,
                    "dollar volume": Strategist_Threshold
                })

           
                current_dollar_value -= Strategist_Threshold
                current_set = []
    break

# Final DataFrame
final_set_df = pd.DataFrame(final_set)
#final_set_df.to_parquet("final_set.parquet")

print(final_set_df)
#2 core machine  currently it will take forever to run with 1000 stocks, how can I minimize the total time?

VERISION 2
import pandas as pd

from concurrent.futures import ThreadPoolExecutor, as_completed

# Load Excel
#df = pd.read_excel("test4stocks.xlsx") #can be changed to whatever type we want 
df = pd.read_parquet("1000Stocks.parquet")
df = df.dropna() # clean empty lines
df["date"] = pd.to_datetime(df["date"]) # converts data from the fprm of YYYY MM DD Time type to pandas  

final_set = [] # empty list for dollar bars

def process_ticker(ticker_df):
    local_final = []
#for ticker, ticker_df in df.groupby("ticker"):  #split per tickers --- apple, nvda etc
    for date, day_df in ticker_df.groupby("date"):  #similarly split by day
        day_df = day_df.sort_values("date").reset_index(drop=True) # making sure the chronological order is fine and every piece starts as 0 index

    #Dollar Volume
        day_df["dollarVolume"] = day_df["close"] * day_df["volume"]

        # 50 bars per day target
        Strategist_Threshold = day_df["dollarVolume"].sum() / 50    #calculate the total dv per day and /50

        current_dollar_value = 0
        current_set = []

        #dollar bar
        for index, row in day_df.iterrows():

            dollar_tot = row["dollarVolume"]
            current_dollar_value += dollar_tot
            current_set.append(row)

            if current_dollar_value >= Strategist_Threshold:

                new_df_dollar_bar = pd.DataFrame(current_set) # new dataframe

                ticker_name = new_df_dollar_bar.iloc[0]["ticker"]
                open_price = new_df_dollar_bar.iloc[0]["close"]
                close_price = new_df_dollar_bar.iloc[-1]["close"]
                high_price = new_df_dollar_bar["close"].max()
                low_price = new_df_dollar_bar["close"].min()

                local_final.append({
                    "ticker": ticker_name,
                    "date": date,
                    "open": open_price,
                    "high": high_price,
                    "low": low_price,
                    "close": close_price,
                    "dollar volume": Strategist_Threshold
                })

           
                current_dollar_value -= Strategist_Threshold
                current_set = []
    return local_final

ticker_groups = [g for _, g in df.groupby("ticker")]

final_results = []

with ThreadPoolExecutor(max_workers=4) as executor:  # adjust threads to CPU cores
    futures = [executor.submit(process_ticker, tg) for tg in ticker_groups]
    from concurrent.futures import as_completed
    for future in as_completed(futures):
        final_results.extend(future.result())

# Final DataFrame
final_set_df = pd.DataFrame(final_results)
final_set_df.to_parquet("final_set.parquet")

print(final_set_df)
#2 core machine 
'''
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# ----------------------------
# 1. Load Parquet
# ----------------------------
df = pd.read_parquet("1000Stocks.parquet")
df = df.dropna(subset=["ticker", "date", "close", "volume"])  # essential columns
df["date"] = pd.to_datetime(df["date"])

# ----------------------------
# 2. Dollar-bar processing
# ----------------------------
def process_ticker(ticker_df):
    """Compute dollar bars for a single ticker safely."""
    results = []
    if ticker_df.empty:
        return results

    ticker_df = ticker_df.sort_values("date").reset_index(drop=True)
    ticker_df["dollarVolume"] = ticker_df["close"] * ticker_df["volume"]

    for date, day_df in ticker_df.groupby(ticker_df["date"].dt.date):
        if day_df.empty:
            continue

        threshold = day_df["dollarVolume"].sum() / 50
        if threshold == 0:
            continue  # skip days with zero volume

        current_dv = 0
        idx_start = 0

        for idx, row in day_df.iterrows():
            current_dv += row["dollarVolume"]

            if current_dv >= threshold:
                bar = day_df.iloc[idx_start:idx+1]
                if not bar.empty:
                    results.append({
                        "ticker": row["ticker"],
                        "date": date,
                        "open": bar.iloc[0]["close"],
                        "high": bar["close"].max(),
                        "low": bar["close"].min(),
                        "close": bar.iloc[-1]["close"],
                        "dollar volume": threshold
                    })
                current_dv -= threshold
                idx_start = idx + 1

    return results

# ----------------------------
# 3. Split by ticker
# ----------------------------
ticker_groups = [group for _, group in df.groupby("ticker")]

# ----------------------------
# 4. Parallel processing
# ----------------------------
final_results = []
NUM_THREADS = 8

with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
    futures = [executor.submit(process_ticker, tg) for tg in ticker_groups]
    for future in as_completed(futures):
        try:
            res = future.result()
            if res:
                final_results.extend(res)
        except Exception as e:
            print(f"Error processing a ticker group: {e}")

# ----------------------------
# 5. Combine results
# ----------------------------
if final_results:
    final_set_df = pd.DataFrame(final_results)
    final_set_df.to_parquet("final_set.parquet", index=False)
    print(final_set_df)
else:
    print("No dollar bars generated.")

     ticker        date     open     high      low     close  dollar volume
0      MSFT  2025-04-21  363.150  363.510  363.150  363.5100   1.387097e+08
1      MSFT  2025-04-21  363.490  364.190  362.625  363.2500   1.387097e+08
2      MSFT  2025-04-21  363.310  363.520  362.295  362.2950   1.387097e+08
3      MSFT  2025-04-21  363.010  363.010  361.900  361.9000   1.387097e+08
4      MSFT  2025-04-21  361.520  361.520  360.475  360.6088   1.387097e+08
...     ...         ...      ...      ...      ...       ...            ...
1227    QQQ  2026-02-05  598.830  604.140  594.970  597.6600   9.498903e+08
1228    QQQ  2026-02-06  599.680  611.290  599.680  609.2600   8.536187e+08
1229    QQQ  2026-02-09  605.887  616.445  605.887  614.6000   6.386890e+08
1230    QQQ  2026-02-10  615.350  616.980  611.070  611.8210   6.207878e+08
1231    QQQ  2026-02-11  617.390  617.390  608.010  613.0200   6.582823e+08

[1232 rows x 7 columns]


In [4]:
'''Specific Tasks:
- For each data frame generated by the data curator, you should build an
optimization loop that is response for:
o Iterate d from 0.0 to 1.0
▪ Pass each d to the FDD function
• Run the Augmented Dickey-Fuller test using the statsmodels
python library
o Work with the back tester to implement their method to find the lowest
possible d where the p-value drops below 0.05
o Pass the data frame and the d value to the backend tester
'''
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
from concurrent.futures import ThreadPoolExecutor, as_completed


#Strategiest's data -- not applicable
#volatile_tickers = ["NVDA", "TSLA", "PLTR", "AMD", "COIN"]
#stable_tickers = ["DUK", "SO", "NEE", "D", "AEP"]

#
final_set_df["date"] = pd.to_datetime(final_set_df["date"])
final_set_df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

#FW fractional diff weights calc
def getWeights_FFD(d, thresh=10**-5):
    w = [1.0] #current day
    k = 1
    while True:
        w_k = -w[-1]*(d-k+1)/k #text book formular
        if abs(w_k)<thresh:
            break # we want to cut off this data
        w.append(w_k) # want to keep data
        k+=1
    return np.array(w[::1]).reshape(-1,1) # reversed order list 

def fracDiff_FFD(series, d, thresh=10**-5):
    series = series.astype(float) # make sure we work with floating format
    w = getWeights_FFD(d, thresh) #sending data to get selected or deleted
    width = len(w)-1 #textbook -- keeping up with indexes

    output = pd.Series(index=series.index, dtype='float64') #creates an empty Panda series to contain results 
    for i in range(width, len(series)):
        window = series.iloc[i-width:i+1]
        if window.isnull().any():
            continue
        output.iloc[i] = np.dot(w.T, window)[0] # computes weighted sum of the window
    return output

def find_min_d(series, d_step=0.02, thresh=10**-5, p_cutoff=0.05): #series = close price, 
    for d in np.arange(0,1+d_step,d_step): #iterate d from 0.0 to 1.0
        diff_series = fracDiff_FFD(series, d, thresh).dropna()
        if len(diff_series)<30: #30 in ADF -- ask strategist if it is okay? should i do 50 instead!!!
            continue
        pval = adfuller(diff_series, maxlag=1, regression='c')[1]
        if pval < p_cutoff: # we want pval<0.05
            return round(d, 4), pval # we want min d
    return None, None # if no d from 0 till 1 "failure"

def process_ticker_fracdiff(ticker_df): # handle one stock at a time
    ticker_df = ticker_df.sort_values("date")
    ticker = ticker_df["ticker"].iloc[0]
    series = ticker_df["close"]

    optimal_d, pval = find_min_d(series) #finds min d for stationarity
    if optimal_d is None: #skipping ticke if no stationary transformation
        return None
    
    stationary_series = fracDiff_FFD(series, optimal_d)
    aligned_original = series.loc[stationary_series.dropna().index]
    corr = np.corrcoef(aligned_original, stationary_series.dropna())[0,1] #measures memory retained

    result = ticker_df.copy()
    result["fracdiff_close"] = stationary_series # new column
    result["optimal_d"] = optimal_d
    result["adf_pvalue"] = pval
    result["memory_retained"] = corr

    return result

ticker_group = [group for _, group in final_set_df.groupby("ticker")]
final_results = []

NUM_THREADS = 8
with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
    futures = [executor.submit(process_ticker_fracdiff, tg) for tg in ticker_groups]
    for future in as_completed(futures):
        res = future.result()
        if res is not None:
            final_results.append(res)

if final_results:
    final_df = pd.concat(final_results).reset_index(drop=True)
    final_df.to_parquet("all_tickers_fracdiff_summary.parquet", index=False)

    summary_df = final_df.groupby("ticker")[["optimal_d", "adf_pvalue","memory_retained"]].first()
    summary_df.to_parquet("all_tickers_fracdiff_summary.parquet")

    print("\n================ FRACTIONAL DIFFERENTIATION RESULTS ================\n")
    print(summary_df)
    print("\n=====================================================================\n")

    print("Fractional differentiation complete for all tickers.")
else:
    print("No stationary series found.")







================ FRACTIONAL DIFFERENTIATION RESULTS ================

        optimal_d  adf_pvalue  memory_retained
ticker                                        
MSFT         0.04    0.012858         0.797576
MU           0.22    0.007122         0.954385
NVDA         0.02    0.049468         0.951451
QQQ          0.00    0.019357         1.000000
TSLA         0.10    0.022626         0.828139


Fractional differentiation complete for all tickers.
